# 🤖 Aşama 3: Transformer Modeli (BERTurk + mT5)

**Hamza'nın görevi — Derin öğrenme modelleri**

Bu notebook'ta:
1. BERTurk (`dbmdz/bert-base-turkish-cased`) duygu analizi için **fine-tune** edilecek
2. mT5 ile **çoklu özetleme** denenecek (zero-shot)
3. Sonuçlar baseline ile karşılaştırılacak

> ⚠️ **MUTLAKA GPU kullanın!** Colab → Çalışma Zamanı → Çalışma Zamanı Türünü Değiştir → **T4 GPU**
> 
> Eğitim CPU'da saatler sürer, GPU'da ~15-30 dakika.

## 0) Kurulum ve GPU Kontrolü

In [ ]:
import sys, os
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB and not os.path.exists('src'):
    !git clone https://github.com/erncanyildirim/metin-ozetleme-ve-duygu-analizi.git
    %cd metin-ozetleme-ve-duygu-analizi
    !pip install -q -r requirements.txt
    !pip install -q datasets accelerate

# GPU kontrolü
import torch
print(f'🖥️  CUDA mevcut mu : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU adı       : {torch.cuda.get_device_name(0)}')
    print(f'   GPU belleği   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('   ⚠️  UYARI: GPU yok! Colab\'da Çalışma Zamanı → T4 GPU seçin.')

## 1) BERTurk Duygu Analizi — Fine-Tuning

**Süre:** ~15-30 dk (T4 GPU, 10K örnek için)

Eğer veri seti çok büyükse, küçük bir alt küme ile başla (sample_size parametresi).

In [ ]:
from src.transformer_model import train_sentiment_model

# Hızlı test için 30000 örnek; tam veri için sample_size=None
model_dir = train_sentiment_model(
    sample_size=30000,  # None yaparsanız tüm veri kullanılır
    num_epochs=3,
    batch_size=16,
)
print(f'✅ Model kaydedildi: {model_dir}')

## 2) BERTurk Çıkarım ve Demo

In [ ]:
from src.transformer_model import load_sentiment_model, predict_sentiment_transformer

sentiment_pipeline = load_sentiment_model()

demo_yorumlar = [
    'Bu ürün kesinlikle harika, çok memnun kaldım',
    'Berbat, hayatımda gördüğüm en kötü ürün',
    'İdare eder, fiyatına göre fena değil',
    'Kargo çok geç geldi ama ürün güzel',  # Karışık
    'Yorumlardan etkilenip aldım, hata yapmışım',  # İronik
]

for yorum in demo_yorumlar:
    res = predict_sentiment_transformer(yorum, sentiment_pipeline)
    print(f"  {res['emoji']} {yorum}")
    print(f"     → {res['sentiment']} (skor: {res['confidence']:.3f})\n")

## 3) BERTurk vs Baseline — Test Setinde Karşılaştırma

In [ ]:
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Test verisini yükle
from src.baseline_model import load_and_prepare_data
X_train, X_test, y_train, y_test, label_names = load_and_prepare_data()

# Eşit karşılaştırma için aynı test setinde:
# Süre kazanmak için ilk 2000 örneği al (BERT yavaş)
test_sample_size = min(2000, len(X_test))
X_test_s = X_test.iloc[:test_sample_size].tolist()
y_test_s = y_test.iloc[:test_sample_size].tolist()

# --- Baseline (SVM) tahminleri ---
svm_model = joblib.load('models/svm_(linearsvc).joblib')
svm_preds = svm_model.predict(X_test_s)

# --- BERTurk tahminleri ---
print('⏳ BERTurk ile tahmin yapılıyor...')
bert_preds = []
for text in X_test_s:
    res = sentiment_pipeline(text[:512])[0]
    label_map = {
        'LABEL_0': 'negatif', 'LABEL_1': 'nötr', 'LABEL_2': 'pozitif',
        'pozitif': 'pozitif', 'negatif': 'negatif', 'nötr': 'nötr',
    }
    bert_preds.append(label_map.get(res['label'], res['label'].lower()))

# --- Metrikler ---
svm_f1 = f1_score(y_test_s, svm_preds, average='macro')
bert_f1 = f1_score(y_test_s, bert_preds, average='macro')
svm_acc = accuracy_score(y_test_s, svm_preds)
bert_acc = accuracy_score(y_test_s, bert_preds)

comparison = pd.DataFrame({
    'Model': ['SVM (Baseline)', 'BERTurk'],
    'Accuracy': [svm_acc, bert_acc],
    'F1 (Macro)': [svm_f1, bert_f1],
})
print('\n📊 KARŞILAŞTIRMA:')
comparison.set_index('Model')

In [ ]:
# Grafik (Rapor Şekil 4.5)
fig, ax = plt.subplots(figsize=(10, 5))
metrics_df = comparison.melt(id_vars='Model', var_name='Metrik', value_name='Skor')
sns.barplot(data=metrics_df, x='Metrik', y='Skor', hue='Model', 
            palette=['#4a90e2', '#dc3545'], ax=ax)
ax.set_title('Baseline (SVM) vs Transformer (BERTurk)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1)
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3)
plt.tight_layout()
plt.savefig('rapor/sekil_4_5_baseline_vs_bert.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix'ler
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, preds) in zip(axes, [('SVM', svm_preds), ('BERTurk', bert_preds)]):
    cm = confusion_matrix(y_test_s, preds, labels=label_names)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(f'{name} — Confusion Matrix')
    ax.set_xlabel('Tahmin')
    ax.set_ylabel('Gerçek')
plt.tight_layout()
plt.savefig('rapor/sekil_4_5b_confusion_compare.png', dpi=150, bbox_inches='tight')
plt.show()

## 4) mT5 ile Çoklu Metin Özetleme

In [ ]:
from src.transformer_model import load_summarization_model, summarize_multiple_reviews

# Modeli yükle (zero-shot kullanım)
summarizer = load_summarization_model()

In [ ]:
# Aynı ürün hakkında 5 yorumu özetle
yorumlar = [
    'Bu telefon gerçekten harika, kamerası süper kaliteli, pil ömrü de uzun.',
    'Telefonun bataryası beklediğimden uzun gitti, bir günden fazla dayanıyor.',
    'Kamerası özellikle gece çekimlerinde çok başarılı, ışık az olunca bile net görüntü.',
    'Fiyat performans olarak çok başarılı, bu paraya bu kadar özellik az bulunur.',
    'Tek olumsuz yanı ekranı biraz küçük geldi, ama genel olarak memnunum.',
]

print('📝 GİRDİ YORUMLAR:')
for i, y in enumerate(yorumlar, 1):
    print(f'   {i}. {y}')

summary = summarize_multiple_reviews(yorumlar, summarizer)
print(f'\n📄 ÜRETİLEN ÖZET:\n   {summary}')

## 5) ROUGE Skorları (Özetleme Değerlendirmesi)

In [ ]:
from src.evaluation import evaluate_summarization_rouge

# Test örnekleri: birden fazla ürün için referans ve tahmin özetleri
test_cases = [
    {
        'yorumlar': [
            'Ürün gerçekten kaliteli, çok memnun kaldım.',
            'Tam beklediğim gibi, tavsiye ederim.',
            'Fiyat performans olarak harika.',
        ],
        'referans': 'Ürün kaliteli, fiyat performans olarak başarılı ve tavsiye edilebilir.',
    },
    {
        'yorumlar': [
            'Kargo geç geldi ve ürün hasarlıydı.',
            'İade etmek zorunda kaldım, kötü deneyim.',
            'Paketleme çok özensizdi, ürün kırık çıktı.',
        ],
        'referans': 'Kargo ve paketleme sorunları yüzünden ürün hasarlı geldi ve iade edildi.',
    },
]

references = []
predictions = []

for case in test_cases:
    pred = summarize_multiple_reviews(case['yorumlar'], summarizer)
    references.append(case['referans'])
    predictions.append(pred)
    print(f'\nReferans: {case["referans"]}')
    print(f'Üretilen: {pred}')

# ROUGE skorları
rouge_results = evaluate_summarization_rouge(references, predictions, model_name='mT5-small')

## 6) Modelleri Drive'a Kaydet (Colab için)

In [ ]:
if IS_COLAB:
    from google.colab import drive
    print('💾 Eğitilmiş modelleri Drive\'a yedeklemek isterseniz:')
    print()
    print('   drive.mount("/content/drive")')
    print('   !cp -r models/berturk-sentiment /content/drive/MyDrive/')
    print()
    print('   # Sonradan tekrar erişmek için:')
    print('   !cp -r /content/drive/MyDrive/berturk-sentiment models/')

## ✅ Sonuç

- BERTurk fine-tune edildi (`models/berturk-sentiment/`)
- mT5-small ile özetleme çalıştırıldı
- Baseline ile karşılaştırma yapıldı

**Rapor için kaydedilecek sayılar:**
- Tablo 4.4.2 — BERTurk Sonuçları
- Tablo 4.4.3 — SVM vs BERTurk karşılaştırması
- Tablo 4.5 — ROUGE skorları (özetleme)
- Şekiller: `rapor/sekil_4_5_*.png`

**Sonraki adım:** Streamlit arayüzünü test et (`streamlit run app.py`)